In [1]:
import pandas as pd
from datetime import timedelta
from sklearn.ensemble import IsolationForest


In [2]:
INPUT_FOLDER = "input"
OUTPUT_FOLDER = "output"

MODULE1_CSV = f"{OUTPUT_FOLDER}/module1_output.csv"

print(f"Loading events from: {MODULE1_CSV}")
events = pd.read_csv(MODULE1_CSV)

# Convert timestamps
events["TimeCreated"] = pd.to_datetime(events["TimeCreated"], errors="coerce")
events = events.rename(columns={"TimeCreated": "Time"})

print("Events loaded:", len(events))
print("\nDataset preview:")
events.head()

Loading events from: output/module1_output.csv
Events loaded: 1934

Dataset preview:


,EventID,Time,SourceLog,Computer,User,ProcessId,Target,CommandLine
0,4624,2025-12-09 04:12:44.955124+00:00,Security.evtx,KRYP10N,KRYP10N$,0x0000000000000488,-,-
1,4672,2025-12-09 04:12:44.955135+00:00,Security.evtx,KRYP10N,SYSTEM,-,-,-
2,4624,2025-12-09 04:15:41.672543+00:00,Security.evtx,KRYP10N,KRYP10N$,0x0000000000000488,-,-
3,4672,2025-12-09 04:15:41.672552+00:00,Security.evtx,KRYP10N,SYSTEM,-,-,-
4,4624,2025-12-09 04:16:39.702091+00:00,Security.evtx,KRYP10N,KRYP10N$,0x0000000000000488,-,-


In [3]:
WINDOW = timedelta(minutes=10)

FEATURE_EVENTS = {
    "proc_create": [4688],
    "logon_success": [4624],
    "logon_fail": [4625],
    "log_cleared": [1102],
    "system_start": [6005],
    "system_shutdown": [6006],
    "priv_assign": [4672],
    "process_install": [4697],
    "time_change": [4616],
}

# Sort timeline
events = events.sort_values("Time").reset_index(drop=True)
start, end = events["Time"].min(), events["Time"].max()

print(f"⏱ Timeline: {start} → {end}")

rows = []
t0 = start

while t0 <= end:
    t1 = t0 + WINDOW
    window = events[(events["Time"] >= t0) & (events["Time"] < t1)]
    
    counts = {name: window["EventID"].isin(ids).sum() 
              for name, ids in FEATURE_EVENTS.items()}
    
    rows.append({"window_start": t0, "window_end": t1, **counts})
    t0 = t1

features = pd.DataFrame(rows)
print("✔ Feature vectors created:", len(features))

features.to_csv(f"{OUTPUT_FOLDER}/module3_features.csv", index=False)
print("📄 Saved: output/module3_features.csv")

features.head()


⏱ Timeline: 2025-05-23 05:26:03.962399+00:00 → 2025-12-12 03:45:20.290403+00:00
✔ Feature vectors created: 29222
📄 Saved: output/module3_features.csv


,window_start,window_end,proc_create,logon_success,logon_fail,log_cleared,system_start,system_shutdown,priv_assign,process_install,time_change
0,2025-05-23 05:26:03.962399+00:00,2025-05-23 05:36:03.962399+00:00,0,0,0,0,1,1,0,0,0
1,2025-05-23 05:36:03.962399+00:00,2025-05-23 05:46:03.962399+00:00,0,0,0,0,0,0,0,0,0
2,2025-05-23 05:46:03.962399+00:00,2025-05-23 05:56:03.962399+00:00,0,0,0,0,0,0,0,0,0
3,2025-05-23 05:56:03.962399+00:00,2025-05-23 06:06:03.962399+00:00,0,0,0,0,0,0,0,0,0
4,2025-05-23 06:06:03.962399+00:00,2025-05-23 06:16:03.962399+00:00,0,0,0,0,0,0,0,0,0


In [5]:
FEAT_FILE = f"{OUTPUT_FOLDER}/module3_features.csv"
df = pd.read_csv(FEAT_FILE)

print("📄 Loaded feature matrix:", df.shape)

X = df.drop(columns=["window_start", "window_end"])

iso = IsolationForest(
    contamination=0.03, 
    n_estimators=200,
    random_state=42
)

print("🧠 Training model...")
iso.fit(X)

print("🔍 Scoring...")
df["score"] = iso.decision_function(X)
df["anomaly_flag"] = (df["score"] < 0).astype(int)

df.to_csv(f"{OUTPUT_FOLDER}/module3_anomalies.csv", index=False)
print("🚨 Saved anomaly results: output/module3_anomalies.csv")

df.head()


📄 Loaded feature matrix: (29222, 11)
🧠 Training model...
🔍 Scoring...
🚨 Saved anomaly results: output/module3_anomalies.csv


,window_start,window_end,proc_create,logon_success,logon_fail,log_cleared,system_start,system_shutdown,priv_assign,process_install,time_change,score,anomaly_flag
0,2025-05-23 05:26:03.962399+00:00,2025-05-23 05:36:03.962399+00:00,0,0,0,0,1,1,0,0,0,-0.125431,1
1,2025-05-23 05:36:03.962399+00:00,2025-05-23 05:46:03.962399+00:00,0,0,0,0,0,0,0,0,0,0.000000,0
2,2025-05-23 05:46:03.962399+00:00,2025-05-23 05:56:03.962399+00:00,0,0,0,0,0,0,0,0,0,0.000000,0
3,2025-05-23 05:56:03.962399+00:00,2025-05-23 06:06:03.962399+00:00,0,0,0,0,0,0,0,0,0,0.000000,0
4,2025-05-23 06:06:03.962399+00:00,2025-05-23 06:16:03.962399+00:00,0,0,0,0,0,0,0,0,0,0.000000,0


In [6]:
anom = pd.read_csv(f"{OUTPUT_FOLDER}/module3_anomalies.csv")

print("📌 Total windows:", len(anom))
print("📌 Anomalies:", anom["anomaly_flag"].sum())

top = anom.sort_values("score").head(20)

print("\n🔎 TOP 20 Most Anomalous Windows:")
top

📌 Total windows: 29222
📌 Anomalies: 347

🔎 TOP 20 Most Anomalous Windows:


,window_start,window_end,proc_create,logon_success,logon_fail,log_cleared,system_start,system_shutdown,priv_assign,process_install,time_change,score,anomaly_flag
29031,2025-12-10 19:56:03.962399+00:00,2025-12-10 20:06:03.962399+00:00,26,120,0,0,2,2,114,0,0,-0.443643,1
29093,2025-12-11 06:16:03.962399+00:00,2025-12-11 06:26:03.962399+00:00,0,16,0,0,0,0,16,0,0,-0.408224,1
29032,2025-12-10 20:06:03.962399+00:00,2025-12-10 20:16:03.962399+00:00,0,16,0,0,0,0,16,0,0,-0.408224,1
29081,2025-12-11 04:16:03.962399+00:00,2025-12-11 04:26:03.962399+00:00,0,14,0,0,0,0,12,0,0,-0.407649,1
28939,2025-12-10 04:36:03.962399+00:00,2025-12-10 04:46:03.962399+00:00,0,14,0,0,0,0,12,0,0,-0.407649,1
28957,2025-12-10 07:36:03.962399+00:00,2025-12-10 07:46:03.962399+00:00,0,13,0,0,0,0,11,0,0,-0.406789,1
28802,2025-12-09 05:46:03.962399+00:00,2025-12-09 05:56:03.962399+00:00,0,9,0,0,0,0,9,0,0,-0.405643,1
28804,2025-12-09 06:06:03.962399+00:00,2025-12-09 06:16:03.962399+00:00,0,9,0,0,0,0,9,0,0,-0.405643,1
29022,2025-12-10 18:26:03.962399+00:00,2025-12-10 18:36:03.962399+00:00,0,9,0,0,0,0,9,0,0,-0.405643,1
29084,2025-12-11 04:46:03.962399+00:00,2025-12-11 04:56:03.962399+00:00,0,10,0,0,0,0,10,0,0,-0.405643,1
